In [1]:
import numpy as np
import rasterio
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

In [ ]:
def create_training_dataset(image_path, labels_path=None):
    """
    Create training dataset from satellite imagery and optional labels
    
    Parameters:
    image_path: Path to multi-band satellite image
    labels_path: Path to labeled water/non-water mask (optional)
    
    Returns:
    X: Feature array
    y: Labels array (if labels_path provided)
    """
    # Read satellite image
    with rasterio.open(image_path) as src:
        # Read all bands
        image = src.read()
        
    # Calculate NDWI (assuming band 3 is green and band 4 is NIR)
    ndwi = (image[2] - image[3]) / (image[2] + image[3])
    
    # Create feature array with all bands plus NDWI
    X = np.vstack([
        image[0].ravel(),  # Blue
        image[1].ravel(),  # Green
        image[2].ravel(),  # Red
        image[3].ravel(),  # NIR
        ndwi.ravel()
    ]).T
    
    # Read labels if provided
    if labels_path:
        with rasterio.open(labels_path) as src:
            labels = src.read(1)
            y = labels.ravel()
        return X, y
    
    return X


In [ ]:
def train_model(X, y, model_type='rf'):
    """
    Train either Random Forest or SVM model
    
    Parameters:
    X: Feature array
    y: Labels array
    model_type: 'rf' for Random Forest or 'svm' for Support Vector Machine
    
    Returns:
    Trained model
    """
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Initialize model
    if model_type == 'rf':
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42
        )
    else:
        model = SVC(kernel='rbf', random_state=42)
    
    # Train model
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    
    print(f"Model Accuracy: {accuracy:.3f}")
    print("\nConfusion Matrix:")
    print(conf_matrix)
    
    return model

In [ ]:
def predict_water(model, image_path, output_path):
    """
    Apply trained model to new imagery
    
    Parameters:
    model: Trained classifier
    image_path: Path to image to classify
    output_path: Where to save results
    """
    # Read and prepare image
    with rasterio.open(image_path) as src:
        image = src.read()
        profile = src.profile
        
    ndwi = (image[2] - image[3]) / (image[2] + image[3])
    X = np.vstack([
        image[0].ravel(),
        image[1].ravel(),
        image[2].ravel(),
        image[3].ravel(),
        ndwi.ravel()
    ]).T
    
    # Predict
    predictions = model.predict(X)
    
    # Reshape to original dimensions
    water_mask = predictions.reshape(image[0].shape)
    
    # Save results
    profile.update(dtype=rasterio.uint8, count=1)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(water_mask.astype(rasterio.uint8), 1)